# Day 01 — System: ChatGPT
**Focus:** Turn-based architecture, context window management, memory feature, tool calling, safety layers

---

## 1. Foundational Concepts

**Turn-level state**: State within a single message exchange (your question + the model's answer). Rebuilt fresh every call since the model itself is stateless.

**Session-level state**: The running history of the *current* conversation thread. Lost once you start a new chat (unless something gets promoted to long-term memory).

**Memory store**: A separate, persistent, cross-session layer — survives even across brand-new chats. Conceptually distinct from both turn-level and session-level state.

**Key distinction**: Memory is NOT an extension of the context window — it's a curated subset that gets *injected* into the context window per turn, similar to a retrieval step.

---

## 2. Tool Calling

- Model is trained (not hardcoded) to recognize *patterns* indicating a tool is needed — e.g., real-time info requests, precise computation, code execution.
- Available tools are described in the system prompt, which is already part of every request's context — no separate "check" step needed.
- Mechanism: model emits a structured tool-call intent mid-generation → execution happens **outside the model**, in the orchestration layer → result is re-injected as a new "turn" → generation resumes.
- Exact internal routing logic (e.g., is there a separate router model) is **not publicly documented**.

---

## 3. Orchestration Layer

- The model = only generates text/tokens. It cannot browse, execute code, or access memory storage directly.
- The orchestration layer = the system *around* the model that:
  - Executes tool calls
  - Fetches/saves memory
  - Coordinates safety checks
  - Manages conversation history and streaming
- Analogy: model = the "brain" that thinks/speaks; orchestration layer = the "hands and legs" that actually do things.

---

## 4. Safety / Guardrails (Layered, not single-point)

End-to-end flow for a harmful request:

1. **Input filter** — classifier scans the message before it reaches the model; clearly harmful content can be blocked here directly.
2. **Model-trained refusal behavior** — baked in via training (RLHF/Model Spec); recognizes harmful intent even with reframing/excuses.
3. **Instruction hierarchy** (if tools/external content involved) — System instructions > User instructions > External content (web pages, emails, documents). Lower-trust sources cannot override higher-trust ones.
4. **Output moderation** — final scan of the generated response before it's shown to the user.

No single layer is sufficient — this is explicitly acknowledged by OpenAI ("no model-level solution fully prevents injection").

**Real-world proof point**: Microsoft Copilot "Reprompt" vulnerability (CVE-2026-24307, disclosed Jan 2026) — a single click could trigger indirect prompt injection leading to data exfiltration. Patched same month.

**Threat reality check**: Per a 2026 international AI safety report, sophisticated attackers still bypass safeguards ~50% of the time given 10 attempts, even on best-defended models.

---

## 5. Memory System — UPDATED via live search (June 2026, "Dreaming V3")

### Old architecture (pre-2026):
- Explicit saved-facts list (user-triggered: "remember I'm vegetarian")
- A lighter background process ("dreaming," since April 2025) that supplemented but couldn't function standalone

### New architecture (Dreaming V3, rolled out June 4, 2026):
- The explicit-list backbone is **gone**. Memory now rests entirely on a background synthesis process across many conversations simultaneously.
- **Self-updating, time-aware memories**: e.g., "You're going to Singapore in July" automatically rewrites to "You went to Singapore in July 2026" once the trip ends — no manual correction needed.
- Users get a **memory summary page**: view, edit, add, or delete what's stored; can disable memory entirely.
- Self-reported (not independently audited) eval improvements: factual recall 67.9%→82.8%, preference adherence 55.3%→71.3%, time-accuracy 52.2%→75.1% (2025→2026).
- **Tension flagged by research**: a 2026 academic study calls this a "personalization-convenience paradox" — the most valued feature is also the least auditable/controllable one. Relevant regulatory pressure: EU AI Act transparency rules effective Aug 2, 2026; a May 2026 lawsuit alleged ad-tracking code embedded in ChatGPT.

### Memory + Deletion
- Deleting a chat does **NOT** automatically delete facts already saved to memory from that chat.
- To fully remove a fact, you must delete it from **every place it appears** — past chats, archived chats, files, AND the memory summary page.
- Why dreaming doesn't auto-clean on chat deletion (inference, not confirmed): time-based updates work because dreaming sees a *new* signal (e.g., a date passing). Chat deletion provides no such signal — the source just disappears, with no explicit "re-check this" trigger.

### Memory retrieval ≈ RAG
- Memory store = knowledge base (like a vector DB in RAG)
- Current user message = query
- Relevant-fact-fetching step = retriever
- Only relevant facts get added to context, not the entire memory store (for cost/latency/relevance reasons)
- Key difference from classic RAG: the "documents" (memory facts) are **self-updating** over time, not static.

---

## 6. Full Request Lifecycle (per message)

```
1. Input safety check (classifier scans message)
2. Context assembly:
   - New message + system prompt + conversation history
   - Memory retrieval (embed query → fetch relevant facts, RAG-style)
   - Available tools list (already in system prompt)
3. Forward pass through transformer (tokenize → embed → attention layers)
4. Model decides: direct answer vs. tool call
5. If tool call: orchestration layer executes tool → result re-injected as new turn → generation resumes
6. Token-by-token (autoregressive) generation, streamed to user as produced
7. Output safety check (parallel or buffered, exact method not public)
8. Session state updated (this turn added to conversation history)
9. [Asynchronous, NOT real-time] Background "dreaming" process may later update long-term memory
```

**Important**: Steps 1–8 happen in real time per message. Step 9 (memory update) happens **later, in the background** — not instantly within the same message.

---

## 7. Streaming + Mid-Generation Self-Correction

- **Output moderation during streaming**: likely a lightweight, parallel classifier scanning chunks as they're generated (explains occasional abrupt cut-offs mid-response). Exact mechanism not publicly confirmed.
- **Model saying "wait, let me reconsider..."**: this is a *trained language behavior*, not a separate safety system. Because of autoregressive generation, the model "sees" its own prior output as context and can naturally course-correct — same way humans revise mid-sentence. Distinct from an external safety system forcibly cutting a response.

---

## 8. Efficiency: Why Not Everything Gets Re-Embedded Every Time

**KV-Caching**: Key-Value vectors (from attention mechanism) for unchanged parts of context (system prompt, tool list, earlier turns) are computed once and cached — not recomputed every message. Only the new incoming message is freshly processed each turn.

This is why long conversations remain usable without redoing the entire computation from scratch every time.

---

## 9. Why Attention Compute & Memory Cost Grow with Conversation Length

- **Compute cost — Quadratic (O(N²))**: every token attends to every other token. 10x more tokens → ~100x more comparisons. This is why very long conversations slow down response generation.
- **Memory cost — Linear**: each token's KV vectors must be stored in GPU memory. Grows proportionally with length, but hits a hard ceiling (GPU memory limit).
- **Consequence**: this combination (compute exploding quadratically + memory hitting hard limits) is the real engineering reason behind context window limits and why truncation/summarization is necessary once a conversation gets long enough — it's not an arbitrary number OpenAI picked.

---

## Quick Interview-Ready Soundbites

- *"Memory is conceptually a retrieval system over a personalized, self-updating knowledge base — similar to RAG, but the documents are facts about the user."*
- *"No single safety layer is sufficient — defense relies on input filtering, trained refusal behavior, instruction hierarchy, and output moderation working together."*
- *"The model only predicts tokens — it has no native ability to browse, execute code, or persist memory. All of that is handled by an orchestration layer around it."*
- *"Context window limits aren't arbitrary — they come from the real compute (quadratic) and memory (linear, capped) costs of attention at scale."*

---

## Open Question to Revisit
Given a stored memory ("user prefers concise answers") that conflicts with current session behavior (long, detailed messages in a technical debugging thread) — where should that conflict resolution logic live in the architecture, and how should it be resolved?